# RemoGen Studio - expressive narration on Colab (Dia by Nari Labs)

Runs Dia (Apache-2.0, 1.6B) here on Colab's GPU. Unlike XTTS, you can write things like
`(laughs)`, `(sighs)`, `(coughs)` directly into your script and Dia will actually perform them -
this is the option for narration that doesn't sound like a flat text-to-speech reading.

Voice cloning works too, but is handled a bit differently than XTTS: your uploaded reference
clip is automatically transcribed (a small Whisper model, included below) since Dia needs the
clip's transcript alongside the audio itself.

**How to use**
1. **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells below **in order**. First run downloads the model (~3-5 GB) - a few minutes.
3. The last cell prints a **URL** and an **API key** - paste both into RemoGen Studio -> **Voice Engine**, **Test connection**, **Save**.
4. Leave this tab open while you generate. Re-running the last cell gives a **new** URL and key every time.

*Note:* Dia is Apache-2.0 licensed (no non-commercial restriction, unlike XTTS/Coqui).

*Honesty note:* voice cloning here trims the replayed reference clip off the start of each
generation automatically, based on how this class of model is documented to behave. This
project's author could not verify that exact behaviour on a real GPU - if a cloned scene's
audio is missing its first second or has an extra second of the reference clip up front,
that's the thing to look at first (see `_trim_wav_start` in `server_dia.py`).

In [ ]:
# 1) Check for a GPU and install the dependencies (first run: ~3-5 minutes)
!nvidia-smi -L || echo ">>> NO GPU: choose Runtime -> Change runtime type -> T4 GPU, then re-run"
!pip -q install git+https://github.com/nari-labs/dia.git
!pip -q install faster-whisper fastapi "uvicorn[standard]" python-multipart

In [ ]:
%%writefile rg_common.py
"""
Shared plumbing for RemoGen's Colab TTS servers (server_xtts.py, server_dia.py,
server_chattts.py). All three expose the SAME contract:
  GET  /health   -> {"status": "ok" | "loading", "model", "device", "gpu"}
  POST /voices   -> multipart "file" -> {"voice_id": "..."}
  POST /tts      -> JSON {"text", "language", "speed", "voice_id"} -> audio/wav
so the Node backend (services/tts/colabProvider.js) never needs to know or
care which model is actually running - only this file's contents differ.
"""
import glob
import hashlib
import hmac
import io
import math
import os
import struct
import subprocess
import wave
from typing import Optional

from fastapi import Header, HTTPException

API_KEY = os.environ.get("RG_API_KEY", "")
FAKE = os.environ.get("RG_FAKE_TTS") == "1"
VOICE_DIR = os.environ.get("RG_VOICE_DIR", "/tmp/rg_voices")
os.makedirs(VOICE_DIR, exist_ok=True)


def require_key(x_api_key: str = Header(default="")):
    if API_KEY and not hmac.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key")


def voice_path(voice_id: str) -> Optional[str]:
    """Resolves a voice_id to its uploaded file, refusing anything that isn't
    a plain hex id - blocks path-traversal tricks like "../../etc/passwd"."""
    if not voice_id or not voice_id.isalnum():
        return None
    matches = glob.glob(os.path.join(VOICE_DIR, f"{voice_id}.*"))
    return matches[0] if matches else None


def save_uploaded_voice(data: bytes, original_filename: str) -> str:
    """Saves an uploaded reference sample, deduped by content hash so
    re-uploading the same file is free, and normalises it to mono 22.05kHz
    (what these models expect) when ffmpeg is available."""
    if not data:
        raise HTTPException(status_code=400, detail="Empty file")
    if len(data) > 50 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Reference audio is over 50 MB")

    voice_id = hashlib.sha1(data).hexdigest()[:16]
    ext = os.path.splitext(original_filename or "")[1].lower() or ".wav"
    raw_path = os.path.join(VOICE_DIR, f"{voice_id}{ext}")
    with open(raw_path, "wb") as fh:
        fh.write(data)

    wav_path = os.path.join(VOICE_DIR, f"{voice_id}.wav")
    if raw_path != wav_path:
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path, "-ac", "1", "-ar", "22050", wav_path],
                check=True, timeout=120,
            )
            os.remove(raw_path)
        except Exception:
            pass  # keep the original file; most loaders can read it directly
    return voice_id


def wav_duration_seconds(path: str) -> float:
    with wave.open(path, "rb") as w:
        return w.getnframes() / w.getframerate()


def apply_speed(input_path: str, output_path: str, speed: float):
    """Re-times narration with ffmpeg's atempo filter. Used uniformly across
    all three engines rather than each model's own (inconsistent, sometimes
    absent) speed control, so behaviour matches what services/tts/http.js's
    caller expects regardless of which Colab engine is running."""
    speed = max(0.5, min(2.0, float(speed or 1.0)))
    if abs(speed - 1.0) < 0.01:
        if input_path != output_path:
            os.replace(input_path, output_path)
        return
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", input_path, "-filter:a", f"atempo={speed}", output_path],
        check=True, timeout=120,
    )


def fake_wav_bytes(text: str) -> bytes:
    """A short sine-wave tone standing in for real audio, so the HTTP
    plumbing (auth, upload, download) can be tested without a GPU or any of
    these (large) models actually installed. Set RG_FAKE_TTS=1 to use this."""
    seconds = max(0.6, 0.4 * len(text.split()))
    rate = 24000
    frames = b"".join(
        struct.pack("<h", int(6000 * math.sin(2 * math.pi * 220 * i / rate))) for i in range(int(seconds * rate))
    )
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(frames)
    return buf.getvalue()


In [ ]:
%%writefile server_dia.py
"""
RemoGen Studio - Dia (Nari Labs) voice server for Google Colab.
See rg_common.py for the shared HTTP contract all three engine servers share.

Dia is a 1.6B, Apache-2.0-licensed dialogue TTS model. Two things make it
worth a separate engine choice over XTTS: it accepts non-verbal tags like
(laughs), (sighs), (coughs) written directly in the script text, and it
supports genuine voice cloning via an audio prompt.

Dia's cloning is more involved than XTTS's: it needs the ACTUAL TRANSCRIPT
of the reference clip alongside the audio (not just the audio alone), and
it generates the reference clip's audio again as part of its output before
continuing into the new text - so the output has to be trimmed by the
reference clip's own duration to get just the new narration. This file
handles both automatically: faster-whisper transcribes your uploaded
sample once at upload time, and the reference clip's length is trimmed
off every generation.

Caveat worth knowing: the exact trim-and-continue behaviour described above
matches how this class of audio-prompted continuation model is documented
to work, but this project's author (an AI assistant) could not verify it
against a real GPU. If a cloned voice's audio sounds like it's missing its
first second or two, or has an extra second of the reference clip at the
start, that's the first thing to adjust - see the TRIM logic below.
"""
import json
import os
import threading
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import Depends, FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel

from rg_common import (
    FAKE, VOICE_DIR, require_key, voice_path, save_uploaded_voice,
    wav_duration_seconds, apply_speed, fake_wav_bytes,
)

MODEL_CHECKPOINT = "nari-labs/Dia-1.6B-0626"
NON_VERBAL_TAGS = (
    "(laughs), (clears throat), (sighs), (gasps), (coughs), (singing), (sings), "
    "(mumbles), (beep), (groans), (sniffs), (claps), (screams), (inhales), (exhales), "
    "(applause), (burps), (humming), (sneezes), (chuckle), (whistles)"
)

state = {"model": None, "whisper": None, "device": "cpu", "gpu": None, "error": None}
synth_lock = threading.Lock()
# voice_id -> {"wav": path, "transcript": "[S1] ..."} - Dia's cloning needs
# both the audio AND its exact transcript, unlike XTTS's audio-only cloning.
voice_transcripts = {}


def load_model():
    try:
        if FAKE:
            state.update(device="fake", gpu="none (fake mode)", model="fake")
            return
        import torch
        from dia.model import Dia

        device = "cuda" if torch.cuda.is_available() else "cpu"
        state["device"] = device
        state["gpu"] = torch.cuda.get_device_name(0) if device == "cuda" else None
        state["model"] = Dia.from_pretrained(MODEL_CHECKPOINT, compute_dtype="float16" if device == "cuda" else "float32")

        # A small Whisper model just to transcribe short (5-10s) reference
        # clips for cloning - not used for the actual narration at all.
        from faster_whisper import WhisperModel
        state["whisper"] = WhisperModel("small.en", device=device, compute_type="float16" if device == "cuda" else "int8")
    except Exception as exc:
        state["error"] = f"{type(exc).__name__}: {exc}"
        raise


@asynccontextmanager
async def lifespan(_app):
    threading.Thread(target=load_model, daemon=True).start()
    yield


app = FastAPI(title="RemoGen Colab TTS - Dia", lifespan=lifespan)

# Persist transcripts across restarts within the same Colab session (state
# dict alone would lose them if this module got reloaded).
TRANSCRIPT_STORE = os.path.join(VOICE_DIR, "dia_transcripts.json")
if os.path.exists(TRANSCRIPT_STORE):
    try:
        voice_transcripts.update(json.load(open(TRANSCRIPT_STORE)))
    except Exception:
        pass


def save_transcripts():
    json.dump(voice_transcripts, open(TRANSCRIPT_STORE, "w"))


@app.get("/health", dependencies=[Depends(require_key)])
def health():
    if state["error"]:
        return JSONResponse(status_code=500, content={"status": "error", "detail": state["error"]})
    return {
        "status": "ok" if (state["model"] is not None or FAKE) else "loading",
        "model": "dia-1.6b" if not FAKE else "fake",
        "device": state["device"],
        "gpu": state["gpu"],
    }


@app.post("/voices", dependencies=[Depends(require_key)])
async def upload_voice(file: UploadFile = File(...)):
    data = await file.read()
    voice_id = save_uploaded_voice(data, file.filename)
    wav = voice_path(voice_id)

    if FAKE:
        voice_transcripts[voice_id] = "[S1] This is a fake transcript for testing."
    elif voice_id not in voice_transcripts:
        if state["whisper"] is None:
            raise HTTPException(status_code=503, detail="Model is still loading - try again shortly")
        segments, _ = state["whisper"].transcribe(wav, language="en")
        transcript = " ".join(seg.text.strip() for seg in segments).strip()
        if not transcript:
            raise HTTPException(status_code=422, detail="Could not transcribe the reference clip - try a clearer sample")
        # Dia requires every input to open with a speaker tag, and treats
        # this transcript as one continuous [S1] turn.
        voice_transcripts[voice_id] = f"[S1] {transcript}"

    save_transcripts()
    return {"voice_id": voice_id}


class TTSRequest(BaseModel):
    text: str
    language: str = "en"
    speed: float = 1.0
    voice_id: Optional[str] = None


@app.post("/tts", dependencies=[Depends(require_key)])
def synthesize(req: TTSRequest):
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="text is empty")
    if state["error"]:
        raise HTTPException(status_code=500, detail=state["error"])
    if state["model"] is None and not FAKE:
        raise HTTPException(status_code=503, detail="Model is still loading")

    # Single-narrator use: always speaker 1. (laughs)-style tags are just
    # part of the scene text already, written in by the scriptwriter/user -
    # Dia recognises them inline, nothing extra to do here for that part.
    generation_text = f"[S1] {text}"

    audio_prompt = None
    trim_seconds = 0.0
    if req.voice_id:
        audio_prompt = voice_path(req.voice_id)
        if not audio_prompt:
            raise HTTPException(status_code=404, detail="Unknown voice_id - upload the voice sample again")
        transcript = voice_transcripts.get(req.voice_id)
        if not transcript:
            raise HTTPException(status_code=422, detail="No transcript stored for this voice - re-upload the sample")
        generation_text = f"{transcript} {generation_text}"
        trim_seconds = wav_duration_seconds(audio_prompt)

    raw_path = os.path.join(VOICE_DIR, f"dia_raw_{threading.get_ident()}_{os.getpid()}.wav")
    final_path = os.path.join(VOICE_DIR, f"dia_out_{threading.get_ident()}_{os.getpid()}.wav")

    if FAKE:
        with open(raw_path, "wb") as fh:
            fh.write(fake_wav_bytes(text))
    else:
        with synth_lock:
            kwargs = {"use_torch_compile": False, "verbose": False}
            if audio_prompt:
                kwargs["audio_prompt"] = audio_prompt
            output = state["model"].generate(generation_text, **kwargs)
            state["model"].save_audio(raw_path, output)

        if trim_seconds > 0:
            _trim_wav_start(raw_path, trim_seconds)

    apply_speed(raw_path, final_path, req.speed)
    with open(final_path, "rb") as fh:
        audio = fh.read()
    for p in (raw_path, final_path):
        if os.path.exists(p):
            os.remove(p)
    return Response(content=audio, media_type="audio/wav")


def _trim_wav_start(path: str, seconds: float):
    """Cuts the first `seconds` off a WAV in place - removes the replayed
    reference clip from a cloned generation's output. See the module
    docstring's caveat about this specific behaviour being unverified."""
    import wave as wave_mod

    with wave_mod.open(path, "rb") as r:
        params = r.getparams()
        frames_to_skip = min(int(seconds * params.framerate), r.getnframes())
        r.readframes(frames_to_skip)
        remaining = r.readframes(r.getnframes() - frames_to_skip)

    with wave_mod.open(path, "wb") as w:
        w.setparams(params)
        w.writeframes(remaining)


In [ ]:
# Start the TTS server + a public tunnel, then print the URL and API key
import os, re, sys, time, json, secrets, subprocess, urllib.request, urllib.error

os.environ["COQUI_TOS_AGREED"] = "1"     # harmless if this engine doesn't use it
API_KEY = secrets.token_urlsafe(24)      # protects the public URL - don't share it
PORT = 8000

def health():
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health", headers={"X-API-Key": API_KEY})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())

# --- start the server (loads the model in the background) ---
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server_dia:app", "--host", "127.0.0.1", "--port", str(PORT)],
    env={**os.environ, "RG_API_KEY": API_KEY},
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
print("Loading Dia (first run downloads the model + a small Whisper model, a few minutes)...")
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open("server.log").read()[-3000:])
        raise SystemExit("Server crashed - see the log above")
    try:
        info = health()
        if info.get("status") == "ok":
            break
    except urllib.error.HTTPError as e:
        if e.code == 500:
            raise SystemExit("Model failed to load: " + e.read().decode()[:1500])
    except Exception:
        pass                                   # server still starting
    if time.time() - t0 > 900:
        raise SystemExit("Timed out waiting for the model to load")
    time.sleep(5)
print("Model ready on:", info.get("gpu") or info.get("device"))

# --- open a public tunnel (Cloudflare quick tunnel: free, no account) ---
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        public_url = m.group(0)
        break
if not public_url:
    print(open("tunnel.log").read()[-2000:])
    raise SystemExit("Could not open a tunnel - see the log above")

print("\n" + "=" * 64)
print(" PASTE THESE INTO RemoGen Studio -> Voice Engine")
print("=" * 64)
print(f" Engine    : Dia")
print(f" Colab URL : {public_url}")
print(f" API key   : {API_KEY}")
print("=" * 64)
print(" Keep this cell running. Stop it (or close the tab) to shut TTS down.\n")

# --- keep the runtime alive while you work ---
try:
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate(); server.terminate()
    print("Stopped.")

### Troubleshooting
- **"NO GPU"** - Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run all.
- **Install fails on `dia`** - it needs a recent `torch`; run `!pip -q install -U torch` first, restart the runtime, then re-run the install cell.
- **A cloned voice sounds cut off at the start, or has extra sound before it starts** - see the "Honesty note" above; the reference-clip trim may need adjusting for your specific clip length.
- **Non-verbal tags like `(laughs)` are read out loud as the word "laughs"** - make sure the tag is written exactly as shown (see the list in `server_dia.py`'s `NON_VERBAL_TAGS`), with the parentheses.
- **RemoGen says the tunnel is down** - Colab stopped (free tier disconnects after idle/limits). Re-run the last cell and paste the new URL + key.